In [1]:
#imports

import numpy as np


import pandas as pd

from rbg.Utilities.crossvalidation import train_test_split
from rbg.ModelSelection.StratifiedKFold import StratifiedKFold
from rbg.ModelSelection.GridSearch import GridSearch
from rbg.Preprocessing.Standard import Standard
from rbg.Classification.KNN import KNN
from rbg.Utilities.ClassificationMetrics import Accuracy, F1Score
from rbg.Unsupervised.PCA import PCA

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context


from ucimlrepo import fetch_ucirepo

dry_bean = fetch_ucirepo(id=602)

from ucimlrepo import fetch_ucirepo 
  
X = dry_bean.data.features 
y = dry_bean.data.targets 
  
# metadata 



df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)

df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})

TARGET = "Class"
FEATURES = ["Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "Eccentricity", "ConvexArea", "EquivDiameter", "Extent", "Solidity",
    "Roundness", "Compactness", "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"]

#display(df[:5])
#display(df['Class'].value_counts())

# Processamento dos dados usando PCA e KNN
## Pré-processamento e Prevenção de Vazamento de Dados (Data Leakage)

- **Padronização das Variáveis** ($Z$-score)**:** Tanto o PCA quanto o KNN dependem diretamente da escala geométrica dos atributos numéricos. Sem a padronização, atributos com ampla variação escalar (como ```Area```, que pode chegar a mais de 1000) dominariam desproporcionalmente o cálculo das distâncias e a orientação dos componentes principais em relação a atributos delimitados (como ```Solidity```, que se situa no intervalo $(0, 1]$).

- **Isolamento Estrito dos Dados de Teste:** O dataset é dividido inicialmente em conjuntos de treino e teste pela função ```train_test_split```. O conjunto de teste é completamente reservado antes de qualquer cálculo estatístico.

- **Prevenção de Vazamento de Informação:** A média ($\mu$) e o desvio padrão ($\sigma$) das variáveis são calculados exclusivamente sobre o conjunto de treino via ```scaler.fit(X_train)```. As transformações (```scaler.transform()```) em ```X_train``` e ```X_test``` utilizam unicamente os parâmetros extraídos do treino, impedindo que a distribuição estatística do conjunto de teste contamine o aprendizado do modelo.

In [3]:
# Separação entre dados de treino e teste:
X = df.drop('Class', axis=1).values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle = True, random_state=42)

In [4]:
# Normalização dos dados:
standard_scaler = Standard()
standard_scaler.fit(X_train)
X_train = standard_scaler.transform(X_train)
# Para evitar vazamento de dados mas manter a padronização entre todos os dados,
# normalizamos os dados do X de teste também pela média e dp dos dados de treino:
X_test = standard_scaler.transform(X_test)

## Redução de Dimensionalidade com PCA (*Principal Component Analysis*)

Existem alguns motivos para aplicarmos o PCA antes do KNN, dentre eles:

- **Maldição da Dimensionalidade (*Curse of Dimensionality*):** Em espaços de alta dimensão, as distâncias entre pares de pontos tendem a se igualar, o que diminui a capacidade da métrica euclidiana usada no KNN. O PCA projeta os dados em um subespaço de menor dimensão, tornando as métricas de distância do KNN mais significativas e computacionalmente eficientes.

- **Eliminação de Colinearidade:** Atributos altamente correlacionados (como ```Area```, ```Perimeter```, ```MajorAxisLength``` e ```MinorAxisLength```) adicionam redundância. O PCA projeta os dados em eixos ortogonais, ou seja, pouco correlacionados, de forma que features "redundantes" acabam não sendo utilizadas.

- **Seleção por Variância Acumulada:** Em vez de escolher um número arbitrário de componentes, define-se uma meta de retenção da variância total de $95\%$. Essa escolha preserva a maior parte da informação estatística essencial do conjunto de dados, descartando apenas variações de baixo impacto e ruídos geométricos de alta dimensão.

- **Integridade do Pipeline de PCA:** A decomposição SVD e a matriz de autovetores são aprendidas apenas sobre o treino padronizado (```pca.fit(X_train_scaled)```). O conjunto de teste padronizado é projetado utilizando a mesma base ortogonal já calculada (```pca.transform(X_test_scaled)```).

In [5]:
pca = PCA(n_components=0.9) # variância acumulada desejada
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Dimensão original das features: {X_train.shape[1]}")
print(f"Componentes mantidos pelo PCA: {pca.components.shape[0]}")
print(f"Variância acumulada: {np.sum(pca.explained_variance_ratio) * 100:.2f}%")

Dimensão original das features: 16
Componentes mantidos pelo PCA: 4
Variância acumulada: 95.04%


## Avaliação Final do Modelo:
O modelo KNN otimizado e reajustado no treino completo é testado uma única vez sobre `X_test_pca`, gerando métricas de generalização isentas de viés.

# Árvores

In [6]:
from rbg.DecisionTree import ClassificationTree, Bagging, RandomForest

import numpy as np


import pandas as pd

from rbg.Utilities import Gini, train_test_split, error_rate

from rbg.ModelSelection import GridSearch, KFold




Faremos agora o uso de árvores para prever os nossos dados. Primeiro, faremos uma separação entre dados de treino e de teste. Em seguida, aplicaremos às árvores de classificação com e sem poda, bagging e Random Forest aos nossos dados. Começaremos primeiro com todos os dados usando os valores das colunas originais, afim de ilustrar o uso da biblioteca. Além do erro calculado com o error_rate, também faremos o OOB da random e bagging tree.

Como o PCA encontrou as features que ajudam a prever melhor, utilizaremos a matriz transformada pelo nosso PCA. Como o dataset original tem 16 features, e ele é muito longo, a árvore ( até mesmo a mais simples, sem poda) demoraria muito para conseguir encontrar os resultados.


Usaremos então justamente "
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)
"


* Validação:

Para árvores, random forest e bagging, usaremos o GridSearch para procurar, em cada um dos três, o melhor hiperparâmetro para eles.

* Comparação de Modelos:

Com os modelos ajustados pelos seus "hiperparâmetros mais ajustados", iremos fazer uma validação cruzada cmo os dados de treino para decidir qual o melhor modelo.

* Avaliação Final:

Dado o modelo escolhido, iremos fazer a predição dos dados de teste, e tomar seu erro.

* Interpretação dos resultados:

Parte destinada a mais considerações sobre escolhas de métricas e trade-off do viés e da variância.



In [7]:


X = df[FEATURES].to_numpy()

y = df[TARGET].to_numpy()



In [8]:
arvore = ClassificationTree()

arvore.fit(X_train_pca,y_train)

y_previsto = arvore.predict(X_test_pca)

erro_arvore_sem_poda = error_rate(y_previsto,y_test)

print("erro da arvore sem poda:",erro_arvore_sem_poda)

erro da arvore sem poda: 0.1325743665075284


Uma vez que, mesmo no mais simples dataset, tivemos um tempo de compilação gigantesco (mais de 10 minutos), 
para os outros tipos de árvore, que são ainda maiores, faremos uma restrição na quantidade de elementos que iremos usar. 

O dataset original tem 13800 linhas. Isso significa que, mesmo com o PCA otimizanndo ( pois limita a 4 colunas e não 16), temos que para cada nível de profundidade da árvore completo, podemos ter até 13000 * 4 verificações, em busca do melhor corte. 

Logo, para os próximos exemplos / buscas/ análise, iremos pegar apenas as primeiro 50000 linhas 

In [9]:
X_treino = X_train_pca[:5000]
X_teste = X_test_pca[:5000]
y_treino = y_train[:5000]
y_teste = y_test[:5000]

In [10]:
arvore_poda = ClassificationTree()

arvore_poda.fit(X_treino,y_treino, prune = True)

y_previsto_poda = arvore_poda.predict(X_teste)

erro_arvore_com_poda = error_rate(y_previsto_poda,y_teste)

print("erro da arvore com poda:",erro_arvore_com_poda)

info = arvore_poda.get_tree_structure_info()

erro da arvore com poda: 0.13918472273228055


Uma vez que o baggin, com 100 árvores, mesmo diminuindo o dataset para apenas 5000 linhas, estava demorando mais de 60 minutos, afim de ser computacionalmente viável, fizemos apenas tanto random, quanto o bagging, considerando apenas 2 ou 3 árvores em cada. 

In [11]:
arvore_random = RandomForest(n_trees = 3)

arvore_random.fit(X_treino,y_treino)

y_previsto_random = arvore_random.predict(X_teste)

erro_arvore_random = error_rate(y_previsto_random,y_teste)

print("erro da arvore random:",erro_arvore_random)

erro da arvore random: 0.40176276165993385


In [12]:
arvorebag = Bagging(n_trees = 3)

arvorebag.fit(X_treino,y_treino)

y_previstobag = arvorebag.predict(X_teste)

erro_arvore_bag = error_rate(y_previstobag,y_teste)

print("erro da bagging :",erro_arvore_bag)

erro da bagging : 0.20455380095482922


## Validação:

Uma vez que árvores (até mesmo as mais simples, sem poda e sem bagging) tem um custo grande para serem montadas de acordo com o algoritmo implementado (a escolha de melhor splitter que passa por todos os valores, etc), e uma vez que as árvores possuem vários parâmetros, então para fazer uma validação otimizada, usaremos apenas os preditores:

- min_samples_split (e não usaremos min_samples_leaf, que ficará padronizado como 1)
- max_depth (e não usaremos max_leaf)
- min_impurity_decrease
- criterion_split (e o nosso "criterion_evaluation" será por tanto error_rate)

Dentro de criterion_choose, também iremos apenas avaliar a moda já que média para problema de classificação não faz muito sentido, e mediana, apesar dos feitojões terem sido numerados, não tem de fato " uma ordem" para tirarmos a mediana.Também deixaremos de fora "min" e "max" pois não necessariamente " as variáveis dummies extremas" são um bom recorte da classifiação.

Também, nos limitaremos a apenas Gini e entropia para serem os critérios de cálculo de impureza.





In [13]:
#Parametros:

p_grid = {
    'max_depth': [5],
    'min_samples_split': [10,50],
    'min_impurity_decrease':[0.1],
    'criterion_split':[ 'Standard','entropy'],
    'criterion_choose':['Standard']

}

grid_arvore = GridSearch( model = ClassificationTree(), param_grid = p_grid, metrics = error_rate, cv = KFold(k = 5, random_state = 42) )

In [14]:
grid_arvore.fit(X_treino,y_treino)

Agora iremos plotar os parâmetros escolhidos pela nossa árvore.

In [15]:
print(grid_arvore.best_params_)

{'max_depth': 5, 'min_samples_split': 10, 'min_impurity_decrease': 0.1, 'criterion_split': 'entropy', 'criterion_choose': 'Standard'}


Para árvores do tipo baggin, que por terem sido implementadas apenas com numpy (logo, sem a biblioteca de paralelização), fazem as árvores localmente. Logo, fazer apenas uma árvore de bagging já é demorado. Portanto, os modelos que iremos testar serão menores ainda. Mudaremos apenas pro bagging a quantidade de árvores utilizadas. Também faremos uma busca de hiperparâmetros dos elementos visto anteriormente, mas com ainda menos elementos, para ser viável computacionalmente.

In [18]:



pbaggin_grid = {
    'max_depth': [7],
    'min_samples_split': [10],
    'min_impurity_decrease':[0.1],
    'criterion_split':[ 'entropy'],
    'criterion_choose':['Standard'],
    'n_trees':[3, 2]

}

grid_bagging = GridSearch( model = Bagging(), param_grid = pbaggin_grid, metrics = error_rate, cv = KFold(k = 5, random_state = 42) )

In [19]:
grid_bagging.fit(X_treino,y_treino)

In [20]:
print(grid_bagging.best_params_)

{'max_depth': 7, 'min_samples_split': 10, 'min_impurity_decrease': 0.1, 'criterion_split': 'entropy', 'criterion_choose': 'Standard', 'n_trees': 3}


Para árvores do tipo baggin, que por terem sido implementadas apenas com numpy (logo, sem a biblioteca de paralelização), fazem as árvores localmente. Logo, fazer apenas uma árvore de bagging já é demorado. Portanto, os modelos que iremos testar serão menores ainda. Utilizaremos apenas o teste de 100 árvores, que é o padrão do baggin, e para ser viável computacionalmente.

Similarmente, também vamos testar, para random forest, apenas 100 árvores e com o critério de escolha de quantidade de parâmetros ser piso da raiz quadrada do número total de preditores. Como a nosso dataset tem apenas 7 preditores, temos que sempre dois preditores serão escolhidos.

In [21]:



p_random_grid = p_grid = {
    'max_depth': [7],
    'min_samples_split': [10],
    'min_impurity_decrease':[0.1],
    'criterion_split':[ 'entropy'],
    'criterion_choose':['Standard'],
    'n_trees':[3,2],
    'factor_selection': [np.sqrt]

}

grid_Random = GridSearch( model = RandomForest(), param_grid = p_random_grid, metrics = error_rate, cv = KFold(k = 5, random_state = 42) )



In [22]:
grid_Random.fit(X_treino,y_treino)

# Seleção de Modelo


Para a seleção de modelos, iremos fazer um K-Fold usando "os campeões dos hiperparâmetros", treinaremos o dataset neles, e escolheremos aquele que teve menor erro de acordo com o error_rate

In [23]:

param_r = grid_Random.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

print(param_r)

{'max_depth': 7, 'min_samples_split': 10, 'min_impurity_decrease': 0.1, 'criterion_split': 'entropy', 'criterion_choose': 'Standard', 'n_trees': 3, 'factor_selection': <ufunc 'sqrt'>}


In [24]:
param_r = grid_Random.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

bagging = Bagging(**param_bagging)
arvore_pos_grid = ClassificationTree(**param_arvore)
random = RandomForest(**param_r)

In [25]:



k_fold = KFold(random_state = 41, k = 5)
k = k_fold.split(X_treino.copy())
lista_erros_arvore =[]
lista_erros_bag=[]
lista_erros_random =[]


for _, (treino_k, teste_k) in enumerate(k):
    X_treino_k, X_teste_k = X_treino[treino_k], X_treino[teste_k]
    y_treino_k, y_teste_k = y_treino[treino_k], y_treino[teste_k]

    random.fit(X_treino_k,y_treino_k)
    bagging.fit(X_treino_k,y_treino_k)
    arvore_pos_grid.fit(X_treino_k,y_treino_k)
    y_pred_random = random.predict(X_teste_k)
    y_pred_bagging = bagging.predict(X_teste_k)
    y_pred_arvore = arvore_pos_grid.predict(X_teste_k)
    lista_erros_arvore.append(error_rate(y_teste_k, y_pred_arvore))
    lista_erros_bag.append(error_rate(y_teste_k, y_pred_bagging))
    lista_erros_random.append(error_rate(y_teste_k, y_pred_random))

erro_medio_a = np.mean(lista_erros_arvore)
erro_medio_b = np.mean(lista_erros_bag)
erro_medio_r = np.mean(lista_erros_random)


print("erro medio da arvore:",erro_medio_a)
print("erro medio do bagging:",erro_medio_b)
print("erro medio do random forest:",erro_medio_r)

erro medio da arvore: 0.5102
erro medio do bagging: 0.1466
erro medio do random forest: 0.3704


# Avaliação Final

Com o nosso modelo ganhador, iremos treinar todos os dados de treino, para aplicar aos nossos dados de teste,
e depois tirar o erro com o error_rate

In [26]:
if erro_medio_b <= erro_medio_a and erro_medio_b <= erro_medio_r:
    print("vencedor: bagging")
    bagging.fit(X_treino,y_treino)
    predicao_final = bagging.predict(X_teste)
elif erro_medio_a <= erro_medio_b and erro_medio_a <= erro_medio_r:
    print("vencedor: arvore")
    arvore_pos_grid.fit(X_treino,y_treino)
    predicao_final = arvore_pos_grid.predict(X_teste)
else:
    random.fit(X_treino,y_treino)
    predicao_final = random.predict(X_teste)

print("previsao final é:",predicao_final)

print("Erro do modelo:",error_rate(predicao_final,y_teste))


vencedor: bagging
previsao final é: ['SIRA' 'SEKER' 'HOROZ' ... 'DERMASON' 'HOROZ' 'CALI']
Erro do modelo: 0.15828130738156443


# Interpretção dos Resultados:

Faz sentido termos que o bagging venceu as árvores normais pois, mesmo tendo poucs árvores, contribui na diminuição da variância. No entando, para essas poucas árvores ( e diversidade de hiperparâmetros ) que conseguimos testar, é de certa forma, curioso ter ganhado do random. Talvez, justamente por já termos feito o PCA, que diminuiu as colunas que a árvore buscaria os potnos de corte, que fez com que o bagging obtivesse um resultado final melhor que o random. 